# Step 2 — Data Preprocessing & Pipeline Construction
### Ransomware Early-Warning System for Small Institutions (ML Module)

This notebook implements the verified preprocessing pipeline for the MLRan ransomware behavioral dataset. Preprocessing is strictly fitted on `X_train` to eliminate data leakage.

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np

# Add project root to sys.path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.preprocessing import (
    MLRanPreprocessor,
    load_raw_mlran_data,
    split_features_and_targets,
    execute_preprocessing_pipeline
)

raw_dir = os.path.join(project_root, 'data', 'raw', 'MLRan')
processed_dir = os.path.join(project_root, 'data', 'processed')
models_dir = os.path.join(project_root, 'models')

print('Environment initialized successfully.')

## 1. Load Raw MLRan Dataset & Metadata
We load the train, test, labels, metadata, and RFE feature name dictionary.

In [ ]:
train_df, test_df, labels_df, meta_df, rfe_dict = load_raw_mlran_data(raw_dir)
print(f'Train raw shape: {train_df.shape}')
print(f'Test raw shape:  {test_df.shape}')
print(f'Total RFE mapped features: {len(rfe_dict)}')

## 2. Separate Features, Targets, and Metadata
Preserves the official split: 3,905 train samples and 975 test samples.

In [ ]:
X_train, y_train, meta_train, X_test, y_test, meta_test = split_features_and_targets(train_df, test_df)

print('X_train shape:', X_train.shape)
print('y_train distribution:\n', y_train.value_counts())
print('\nX_test shape:', X_test.shape)
print('y_test distribution:\n', y_test.value_counts())

## 3. Data Quality & Preprocessing Checks
- Checking for missing values
- Checking for infinite values
- Checking for constant features

In [ ]:
print('Missing in X_train:', X_train.isna().sum().sum())
print('Missing in X_test: ', X_test.isna().sum().sum())
print('Infinite in X_train:', np.isinf(X_train.values).sum())
print('Infinite in X_test: ', np.isinf(X_test.values).sum())

variances = X_train.var(axis=0)
zero_var = variances[variances == 0].index.tolist()
print(f'Constant/Zero-variance features in X_train: {len(zero_var)}')

## 4. Execute and Save Preprocessing Pipeline
Fits preprocessors (Tree-unscaled & Linear-scaled) strictly on `X_train` and serializes artifacts.

In [ ]:
results = execute_preprocessing_pipeline(raw_dir, processed_dir, models_dir)
for k, v in results.items():
    print(f'{k}: {v}')